# Linking results
### Author: Oscar Heath
### Date: Oct 12 2026

This script takes in as input the `pairs.parquet` file produced from embed_findings.ipynb, and runs all pairs through the `deepseek-v4-flash` LLM to determine if they are linked.

In [9]:
from openai import OpenAI
from dotenv import load_dotenv
import polars as pl
import os
load_dotenv()

True

In [ ]:
from openai import AsyncOpenAI
from openai.providers import bedrock
from dotenv import load_dotenv
import asyncio
import os
load_dotenv()

BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# retries with exponential backoff on rate limits and transient errors
client = AsyncOpenAI(base_url=BASE_URL, max_retries=5)

In [ ]:
MODEL = "deepseek.v3.2"

# price in $ per 1M tokens
INPUT = 0.62
OUTPUT = 1.85

# maximum number of requests in flight at once; on Windows the Jupyter event loop uses
# select(), which caps out at 512 sockets total (including the kernel's own), so keep this well under that
MAX_CONCURRENT = 250

async def query_hypothesis(prompt):
    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
        except Exception as e:
            # leave the row empty so it is retried on the next run
            print(f"request failed: {e}")
            return {"response": "", "input_tokens": 0, "output_tokens": 0, "cost": 0.0}
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = (output_tokens * OUTPUT + input_tokens * INPUT) / 1000000
    return {
        "response": response.choices[0].message.content or "",
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": cost,
    }

In [13]:
df = pl.read_parquet("../../data/hypotheses_pairs.parquet")
def build_prompt(row):
    return f"""You will be given two statistical findings identified from metadata of SRA biosamples. Your task is to determine if the two findings seem related, or refer to the same finding. Respond only with a single word answer: "Yes" or "No".
Finding 1: {row['result_1']}, hypothesis: {row['hypothesis_1']}
Finding 2: {row['result_2']}, hypothesis: {row['hypothesis_2']}
"""
prompts_df = df.with_columns(pl.struct(["result_1", "hypothesis_1", "result_2", "hypothesis_2"]).map_elements(build_prompt, return_dtype=pl.String).alias("prompt"))
prompts_list = prompts_df.select("prompt").to_series().to_list()

In [14]:
prompts_df

row_1,row_2,result_1,result_2,hypothesis_1,hypothesis_2,prompt
i64,i64,str,str,str,str,str
0,53867,"""Samples with the following att…","""Samples with the following att…","""A plausible hypothesis is that…","""A plausible hypothesis is that…","""You will be given two statisti…"
0,3,"""Samples with the following att…","""Samples with the following att…","""A plausible hypothesis is that…","""A plausible hypothesis is that…","""You will be given two statisti…"
0,44723,"""Samples with the following att…","""Samples with the following att…","""A plausible hypothesis is that…","""A plausible hypothesis is that…","""You will be given two statisti…"
1,31092,"""Samples with the following att…","""Samples with the following att…","""A plausible hypothesis is that…","""A plausible hypothesis is that…","""You will be given two statisti…"
1,51928,"""Samples with the following att…","""Samples with the following att…","""A plausible hypothesis is that…","""One plausible hypothesis is th…","""You will be given two statisti…"
…,…,…,…,…,…,…
60721,60720,"""Samples with the following att…","""Samples with the following att…","""One hypothesis is that **HPV16…","""A plausible hypothesis is that…","""You will be given two statisti…"
60721,31639,"""Samples with the following att…","""Samples with the following att…","""One hypothesis is that **HPV16…","""A plausible hypothesis is that…","""You will be given two statisti…"
60722,60720,"""Samples with the following att…","""Samples with the following att…","""A plausible hypothesis is that…","""A plausible hypothesis is that…","""You will be given two statisti…"


# Run all prompts through the model

For each pair, call the model and record whether it judged the two findings related ("Yes") or not ("No") as a boolean. Runs concurrently with retries, and checkpoints progress so an interrupted run over the ~182K pairs can resume where it left off.

In [15]:
# add in columns for response, input tokens, output tokens, and cost
prompts_df = prompts_df.with_columns(
    pl.lit("").alias("response"),
    pl.lit(0).alias("input_tokens"),
    pl.lit(0).alias("output_tokens"),
    pl.lit(0.0).alias("cost")
)

In [ ]:
CHECKPOINT_PATH = "../../data/link_findings_checkpoint.parquet"
CHECKPOINT_EVERY = 1000

# number of rows to query this run, set to None to run on all remaining rows
N_ROWS = None

semaphore = asyncio.Semaphore(MAX_CONCURRENT)
result_cols = ["response", "input_tokens", "output_tokens", "cost"]
if "row_idx" not in prompts_df.columns:
    prompts_df = prompts_df.with_row_index("row_idx")

# resume from the checkpoint if one exists
if os.path.exists(CHECKPOINT_PATH):
    prompts_df = prompts_df.update(pl.read_parquet(CHECKPOINT_PATH), on="row_idx")

todo = prompts_df.filter(pl.col("response") == "")
if N_ROWS is not None:
    todo = todo.head(N_ROWS)
print(f"{prompts_df.height - prompts_df.filter(pl.col('response') == '').height} rows already done, querying {todo.height} rows")

for start in range(0, todo.height, CHECKPOINT_EVERY):
    chunk = todo.slice(start, CHECKPOINT_EVERY)
    results = await asyncio.gather(*(query_hypothesis(prompt) for prompt in chunk["prompt"]))

    results_df = pl.DataFrame(results, schema={col: prompts_df.schema[col] for col in result_cols}).with_columns(chunk["row_idx"])
    prompts_df = prompts_df.update(results_df, on="row_idx")

    # save every completed row so far, failed rows are left out and retried next run
    prompts_df.filter(pl.col("response") != "").select(["row_idx", *result_cols]).write_parquet(CHECKPOINT_PATH)
    print(f"checkpointed {start + chunk.height}/{todo.height} rows, total cost so far: ${prompts_df['cost'].sum():.4f}")

20 rows already done, querying 1000 rows
checkpointed 1000/1000 rows, total cost so far: $0.3009


In [17]:
# parse the Yes/No answer into a boolean, rows not yet queried are left null
answer = pl.col("response").str.strip_chars().str.strip_chars(".\"'*").str.to_lowercase()
prompts_df = prompts_df.with_columns(
    pl.when(answer == "yes").then(True)
    .when(answer == "no").then(False)
    .otherwise(None)
    .alias("findings_related")
)

unparsed = prompts_df.filter((pl.col("response") != "") & pl.col("findings_related").is_null())
print(f"{unparsed.height} responses were not a clear Yes/No")
print(f"total cost: ${prompts_df['cost'].sum():.4f}")

0 responses were not a clear Yes/No
total cost: $0.3009


In [18]:
prompts_df.write_parquet("../../data/hypotheses_linked_findings.parquet")

In [10]:
# get rows where findings are related
related_df = prompts_df.filter(pl.col("findings_related") == True)

In [11]:
related_df.write_parquet("../../data/hypotheses_related_findings.parquet")